In [20]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

import librosa

In [21]:
data_dir = "../Data/genres_original" 
classes = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

In [22]:
def get_train_test_files(data_dir, classes, test_size=0.2):
    train_files, test_files, train_labels, test_labels = [], [], [], []
    
    for i_class, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        # Lấy danh sách toàn bộ đường dẫn file của class hiện tại
        files = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.endswith('.wav')]
        
        # Chia train/test dựa trên danh sách file
        X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
            files, [i_class]*len(files), test_size=test_size, random_state=42
        )
        
        train_files.extend(X_train_f)
        test_files.extend(X_test_f)
        train_labels.extend(y_train_f)
        test_labels.extend(y_test_f)
        
    return train_files, test_files, train_labels, test_labels

In [23]:
train_files, test_files, train_labels, test_labels = get_train_test_files(data_dir, classes)
print(f"Số lượng file train: {len(train_files)} | Số lượng file test: {len(test_files)}")

Số lượng file train: 800 | Số lượng file test: 200


In [24]:
train_files[:5], train_labels[:5]

(['../Data/genres_original\\blues\\blues.00055.wav',
  '../Data/genres_original\\blues\\blues.00088.wav',
  '../Data/genres_original\\blues\\blues.00026.wav',
  '../Data/genres_original\\blues\\blues.00042.wav',
  '../Data/genres_original\\blues\\blues.00069.wav'],
 [0, 0, 0, 0, 0])

In [ ]:
def process_audio_files(file_list, label_list, target_shape=(128, 173)):
    data, labels = [], []
    
    for file_path, label in zip(file_list, label_list):
        try:
            audio_data, sample_rate = librosa.load(file_path, sr=None)
        except Exception as e:
            print(f"Lỗi khi đọc file {file_path}: {e}") # Thêm dòng này để debug
            continue
            
        chunk_samples = 4 * sample_rate
        overlap_samples = 2 * sample_rate
        num_chunks = int(np.ceil((len(audio_data) - chunk_samples) / (chunk_samples - overlap_samples))) + 1

        for i in range(num_chunks):
            start = i * (chunk_samples - overlap_samples)
            end = start + chunk_samples
            chunk = audio_data[start:end]
            
            if len(chunk) < chunk_samples:
                continue

            mel = librosa.feature.melspectrogram(y=chunk, sr=sample_rate, n_mels=128)
            mel_db = librosa.power_to_db(mel, ref=np.max)
            mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

            # 4. Fix kích thước ma trận mà không làm méo hình (Truncate hoặc Padding)
            if mel_db.shape[1] >= target_shape[1]:
                mel_db = mel_db[:, :target_shape[1]]
            else:
                pad_width = target_shape[1] - mel_db.shape[1]
                mel_db = np.pad(mel_db, pad_width=((0,0), (0,pad_width)), mode='constant')

            data.append(np.expand_dims(mel_db, axis=-1))
            labels.append(label)

    return np.array(data), np.array(labels)

In [28]:
print("Đang xử lý tập Train...")
X_train, y_train = process_audio_files(train_files, train_labels)

Đang xử lý tập Train...


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10196\4204671203.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_data, sample_rate = librosa.load(file_path, sr=None)
d:\CondaEnvs\tensorflow_env\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Lỗi khi đọc file ../Data/genres_original\jazz\jazz.00054.wav: 


In [29]:
X_train

array([[[[0.8422373 ],
         [0.8574382 ],
         [0.7786118 ],
         ...,
         [0.7018517 ],
         [0.6420374 ],
         [0.61148834]],

        [[0.8789085 ],
         [0.8915197 ],
         [0.77458   ],
         ...,
         [0.8815767 ],
         [0.84467363],
         [0.77668226]],

        [[0.85273457],
         [0.8406719 ],
         [0.7628252 ],
         ...,
         [0.904792  ],
         [0.90173894],
         [0.84280986]],

        ...,

        [[0.        ],
         [0.        ],
         [0.        ],
         ...,
         [0.        ],
         [0.        ],
         [0.        ]],

        [[0.        ],
         [0.        ],
         [0.        ],
         ...,
         [0.        ],
         [0.        ],
         [0.        ]],

        [[0.        ],
         [0.        ],
         [0.        ],
         ...,
         [0.        ],
         [0.        ],
         [0.        ]]],


       [[[0.5320841 ],
         [0.49972272],
         [0.46

In [30]:
print("Đang xử lý tập Test...")
X_test, y_test = process_audio_files(test_files, test_labels)

Đang xử lý tập Test...


In [31]:
# One-hot encoding
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes=len(classes))
y_test_cat = to_categorical(y_test, num_classes=len(classes))

In [32]:
y_train_cat

array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]], dtype=float32)

In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2

def build_optimized_model(input_shape=(128, 173, 1), num_classes=10):
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape, kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        # Block 2
        Conv2D(64, (3, 3), activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        # Block 3
        Conv2D(128, (3, 3), activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        # Thay vì Flatten, dùng Global Average Pooling để giảm đột biến số lượng tham số
        GlobalAveragePooling2D(),
        
        # Lớp phân loại
        Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model



In [34]:
model = build_optimized_model()

In [35]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 126, 171, 32)      320       
                                                                 
 batch_normalization (BatchN  (None, 126, 171, 32)     128       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2D  (None, 63, 85, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 61, 83, 64)        18496     
                                                                 
 batch_normalization_1 (Batc  (None, 61, 83, 64)       256       
 hNormalization)                                                 
                                                        

In [36]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Tự động dừng nếu val_loss không giảm sau 10 epoch, và phục hồi lại trọng số tốt nhất
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# Tự động giảm Learning Rate đi một nửa nếu val_loss đi ngang sau 5 epoch
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

training_history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=50, 
    batch_size=32,
    callbacks=[early_stopping, reduce_lr]
)

Epoch 1/50
350/350 [==============================] - 166s 469ms/step - loss: 1.6477 - accuracy: 0.5130 - val_loss: 9.3455 - val_accuracy: 0.1001 - lr: 0.0010
Epoch 2/50
350/350 [==============================] - 160s 456ms/step - loss: 1.1515 - accuracy: 0.6942 - val_loss: 1.9262 - val_accuracy: 0.4853 - lr: 0.0010
Epoch 3/50
350/350 [==============================] - 185s 529ms/step - loss: 0.9540 - accuracy: 0.7554 - val_loss: 3.7312 - val_accuracy: 0.2781 - lr: 0.0010
Epoch 4/50
350/350 [==============================] - 173s 493ms/step - loss: 0.8216 - accuracy: 0.7972 - val_loss: 1.2441 - val_accuracy: 0.6505 - lr: 0.0010
Epoch 5/50
350/350 [==============================] - 161s 459ms/step - loss: 0.7223 - accuracy: 0.8313 - val_loss: 2.7547 - val_accuracy: 0.3706 - lr: 0.0010
Epoch 6/50
350/350 [==============================] - 164s 468ms/step - loss: 0.6301 - accuracy: 0.8669 - val_loss: 2.3978 - val_accuracy: 0.4278 - lr: 0.0010
Epoch 7/50
350/350 [==========================

In [37]:
model.save("second_genre_classification_model.h5")

In [42]:
import json
import numpy as np

# Tạo một Encoder tùy chỉnh
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

# Sử dụng tham số 'cls' để gọi Encoder này khi dump
with open("second_training_history.json", "w") as f:
    json.dump(training_history.history, f, cls=NumpyEncoder)
    
print("Đã lưu file thành công!")

Đã lưu file thành công!
